### Multi-Disease GEO Data Collection
**Project:** Pan-Autoimmune miRNA-mRNA Dysregulation Network Analysis  
**Author:** Marjia Islam Tia  
**Date:** May 2025  

#### Objective
Download and inspect raw gene expression datasets from NCBI GEO for four autoimmune diseases:
- Vitiligo (GSE65127)
- Systemic Lupus Erythematosus (GSE318067)
- Rheumatoid Arthritis (GSE93272)
- Type 1 Diabetes (GSE55098)

Key genes of interest: PTPN22, NLRP1

In [1]:
import GEOparse
import pandas as pd
import os

# Set data path relative to project root
os.makedirs("../data/raw", exist_ok=True)

print("Libraries loaded successfully")
print(f"GEOparse version: {GEOparse.__version__}")

Libraries loaded successfully
GEOparse version: 2.0.4


In [5]:
data_raw = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/raw")
os.makedirs(data_raw, exist_ok=True)

gse65127 = GEOparse.get_GEO(geo="GSE65127", destdir=data_raw, silent=False, how="full")

23-May-2026 04:19:56 DEBUG utils - Directory /Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/data/raw already exists. Skipping.
23-May-2026 04:19:56 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE65nnn/GSE65127/soft/GSE65127_family.soft.gz to /Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/data/raw/GSE65127_family.soft.gz
100%|██████████| 23.2M/23.2M [00:34<00:00, 710kB/s]   
23-May-2026 04:20:33 DEBUG downloader - Size validation passed
23-May-2026 04:20:33 DEBUG downloader - Moving /var/folders/rr/fr433gb95fv28s7v4vxmpzsc0000gn/T/tmpp0q7v8ow to /Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/data/raw/GSE65127_family.soft.gz
23-May-2026 04:20:33 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE65nnn/GSE65127/soft/GSE65127_family.soft.gz
23-May-2026 04:20:33 INFO GEOparse - Parsing /Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/data/raw/GSE65127_family.soft.gz: 
23-May-2026 04:20:33 DEBUG GEOparse - DATABASE: GeoMiame
23

In [6]:
# Extract expression matrix from all samples
expr_matrix = gse65127.pivot_samples("VALUE")

print(expr_matrix.shape)
print(expr_matrix.head())

(54675, 40)
name       GSM1587709  GSM1587710  GSM1587711  GSM1587712  GSM1587713  \
ID_REF                                                                  
1007_s_at     11.9879     11.6747     11.3387     10.8127     11.6796   
1053_at        9.1995      9.1957      8.9316      8.9124      9.0747   
117_at         8.2743      7.5336      7.0261      7.6071      7.1794   
121_at         7.7718      6.9443      7.3272      6.8151      6.9986   
1255_g_at      2.9100      3.2325      2.8095      3.0103      3.7755   

name       GSM1587714  GSM1587715  GSM1587716  GSM1587717  GSM1587718  ...  \
ID_REF                                                                 ...   
1007_s_at     11.3450     11.3883     11.6384     11.6478     11.4500  ...   
1053_at        9.1090      8.8098      8.9686      9.2861      8.7077  ...   
117_at         7.0954      7.8685      7.2268      8.6186      6.8247  ...   
121_at         6.8921      7.1245      6.8628      6.9369      7.5555  ...   
1255_g_a

In [7]:
# Get the annotation table from GPL570
gpl = gse65127.gpls["GPL570"]
annotation = gpl.table

print(annotation.shape)
print(annotation.columns.tolist())
print(annotation.head())

(54675, 16)
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
          ID  GB_ACC SPOT_ID Species Scientific Name Annotation Date  \
0  1007_s_at  U48705     NaN            Homo sapiens     Oct 6, 2014   
1    1053_at  M87338     NaN            Homo sapiens     Oct 6, 2014   
2     117_at  X51757     NaN            Homo sapiens     Oct 6, 2014   
3     121_at  X69699     NaN            Homo sapiens     Oct 6, 2014   
4  1255_g_at  L36861     NaN            Homo sapiens     Oct 6, 2014   

       Sequence Type                  Sequence Source  \
0  Exemplar sequence  Affymetrix Proprietary Database   
1  Exemplar sequence                          GenBank   
2  Exemplar sequence  Affymetrix Proprietary 

In [8]:
# Keep only ID and Gene Symbol columns
gene_map = annotation[["ID", "Gene Symbol"]].copy()
gene_map = gene_map.dropna(subset=["Gene Symbol"])
gene_map = gene_map[gene_map["Gene Symbol"] != ""]

print(gene_map.shape)
print(gene_map.head())

(45782, 2)
          ID       Gene Symbol
0  1007_s_at  DDR1 /// MIR4640
1    1053_at              RFC2
2     117_at             HSPA6
3     121_at              PAX8
4  1255_g_at            GUCA1A


In [9]:
# Keep only unambiguous probes (no /// in gene symbol)
gene_map_clean = gene_map[~gene_map["Gene Symbol"].str.contains("///")]

print(f"Before cleaning: {gene_map.shape[0]} probes")
print(f"After cleaning: {gene_map_clean.shape[0]} probes")
print(gene_map_clean.head())

Before cleaning: 45782 probes
After cleaning: 42986 probes
          ID Gene Symbol
1    1053_at        RFC2
2     117_at       HSPA6
3     121_at        PAX8
4  1255_g_at      GUCA1A
6    1316_at        THRA


In [10]:
# Merge expression matrix with gene annotation
expr_matrix_reset = expr_matrix.reset_index()

merged = expr_matrix_reset.merge(gene_map_clean, left_on="ID_REF", right_on="ID", how="inner")

# Set gene symbol as index
merged = merged.drop(columns=["ID_REF", "ID"])
merged = merged.set_index("Gene Symbol")

print(merged.shape)
print(merged.head())

(42986, 40)
             GSM1587709  GSM1587710  GSM1587711  GSM1587712  GSM1587713  \
Gene Symbol                                                               
RFC2             9.1995      9.1957      8.9316      8.9124      9.0747   
HSPA6            8.2743      7.5336      7.0261      7.6071      7.1794   
PAX8             7.7718      6.9443      7.3272      6.8151      6.9986   
GUCA1A           2.9100      3.2325      2.8095      3.0103      3.7755   
THRA             6.2745      8.3758      8.8078      8.2164      7.8973   

             GSM1587714  GSM1587715  GSM1587716  GSM1587717  GSM1587718  ...  \
Gene Symbol                                                              ...   
RFC2             9.1090      8.8098      8.9686      9.2861      8.7077  ...   
HSPA6            7.0954      7.8685      7.2268      8.6186      6.8247  ...   
PAX8             6.8921      7.1245      6.8628      6.9369      7.5555  ...   
GUCA1A           3.0674      3.2138      2.7715      5.0570   

In [11]:
# Average expression values for duplicate genes
merged_clean = merged.groupby("Gene Symbol").mean()

print(f"Before deduplication: {merged.shape[0]} genes")
print(f"After deduplication: {merged_clean.shape[0]} genes")
print(merged_clean.head())

Before deduplication: 42986 genes
After deduplication: 21655 genes
             GSM1587709  GSM1587710  GSM1587711  GSM1587712  GSM1587713  \
Gene Symbol                                                               
A1BG             5.7121     4.86650     4.74210     5.23090     6.35340   
A1BG-AS1         7.1417     6.73460     6.97280     6.88620     7.36990   
A1CF             2.7676     2.59245     2.84865     2.87215     3.12785   
A2M              8.6835     8.65530     8.39330     8.80270     8.85825   
A2M-AS1          5.1971     6.39250     6.26180     7.36640     5.53310   

             GSM1587714  GSM1587715  GSM1587716  GSM1587717  GSM1587718  ...  \
Gene Symbol                                                              ...   
A1BG             5.5152      5.1787     5.58610      5.6677     5.17220  ...   
A1BG-AS1         7.1545      7.3900     7.43460      7.1104     7.01880  ...   
A1CF             2.7096      2.6871     2.72990      2.6551     2.69235  ...   
A2M    

In [12]:
# Verify no duplicates remain
print(merged_clean.index.duplicated().sum())

0


In [14]:
# Save processed GSE65127 expression matrix
data_processed = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/processed")
os.makedirs(data_processed, exist_ok=True)

merged_clean.to_csv(f"{data_processed}/GSE65127_vitiligo_expr.csv")
print(f"Saved: GSE65127_vitiligo_expr.csv")
print(f"Shape: {merged_clean.shape}")

Saved: GSE65127_vitiligo_expr.csv
Shape: (21655, 40)


In [20]:
def process_geo_dataset(geo_id, disease_name, data_raw, data_processed):
    print(f"\nProcessing {geo_id} — {disease_name}")
    
    # Download with fallback
    try:
        gse = GEOparse.get_GEO(geo=geo_id, destdir=data_raw, silent=True, how="full")
    except Exception as e:
        print(f"First attempt failed: {e}")
        print("Retrying...")
        gse = GEOparse.get_GEO(geo=geo_id, destdir=data_raw, silent=False, how="full")
    
    # Expression matrix
    expr = gse.pivot_samples("VALUE")
    
    # Get annotation
    gpl_id = list(gse.gpls.keys())[0]
    annotation = gse.gpls[gpl_id].table
    gene_map = annotation[["ID", "Gene Symbol"]].copy()
    gene_map = gene_map.dropna(subset=["Gene Symbol"])
    gene_map = gene_map[gene_map["Gene Symbol"] != ""]
    gene_map = gene_map[~gene_map["Gene Symbol"].str.contains("///")]
    
    # Merge and clean
    expr_reset = expr.reset_index()
    merged = expr_reset.merge(gene_map, left_on="ID_REF", right_on="ID", how="inner")
    merged = merged.drop(columns=["ID_REF", "ID"])
    merged = merged.set_index("Gene Symbol")
    merged = merged.groupby("Gene Symbol").mean()
    
    # Save
    filename = f"{geo_id}_{disease_name}_expr.csv"
    merged.to_csv(f"{data_processed}/{filename}")
    print(f"Saved: {filename} — Shape: {merged.shape}")
    
    return merged

In [ ]:
## GSE93272 and GSE55098 downloaded manually via curl due to GEOparse FTP instability
## Files saved to data/raw/ then loaded from disk by GEOparse

# datasets = [
#    ("GSE318067", "SLE"),
#    ("GSE93272", "RA"),
#    ("GSE55098", "T1D")
#]

results = {}
for geo_id, disease_name in datasets:
    results[geo_id] = process_geo_dataset(geo_id, disease_name, data_raw, data_processed)


Processing GSE318067 — SLE


/Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/mirna_env/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Saved: GSE318067_SLE_expr.csv — Shape: (21655, 82)

Processing GSE93272 — RA


/Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/mirna_env/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Saved: GSE93272_RA_expr.csv — Shape: (21655, 275)

Processing GSE55098 — T1D


/Users/mohammadabuhuzaifa/Pan-Autoimmune-miRNA-ML/mirna_env/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Saved: GSE55098_T1D_expr.csv — Shape: (21655, 22)
